# Classification Results — Iris

Notebook de consolidación: reproduce el protocolo de `Notebooks/Classification/`
(3 clásicos + 3 cuánticos), guarda métricas y genera figuras para el documento y la defensa.

**Protocolo común**
- Dataset: Iris (3 clases, 4 features)
- Split estratificado: `test_size=0.4`, `random_state=123`
- Escalado: `MinMaxScaler` a `[0, π]`
- Métricas: Accuracy, Precision, Recall, F1, AUC (macro)

**Modelos**
| Familia | Modelo |
|---|---|
| Clásico | SVM RBF, Random Forest, XGBoost |
| Cuántico | QSVM (ZZ kernel), QNN (PennyLane+Torch), VQC |


## Librerías y rutas


In [ ]:
# 1. Imports
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from sklearn import datasets
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
from pennylane.optimize import NesterovMomentumOptimizer
from pennylane import numpy as pnp

# Rutas de salida (funciona desde la raíz del repo o desde Notebooks/Results)
cwd = Path.cwd()
if cwd.name == "Results":
    RESULTS_DIR = cwd
elif (cwd / "Notebooks" / "Results").exists():
    RESULTS_DIR = cwd / "Notebooks" / "Results"
else:
    RESULTS_DIR = cwd
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("FIGURES_DIR:", FIGURES_DIR)
print("METRICS_DIR:", METRICS_DIR)

# Paleta UNIR
UNIR_NAVY = "#0E4A75"
UNIR_AZUL = "#219BC3"
UNIR_GRIS = "#858591"
UNIR_CLARO = "#C0C0C8"

CLASSICAL_MODELS = ["SVM", "Random Forest", "XGBoost"]
QUANTUM_MODELS = ["QSVM", "QNN", "VQC"]
ALL_MODELS = CLASSICAL_MODELS + QUANTUM_MODELS

MODEL_COLORS = {
    "SVM": UNIR_NAVY,
    "Random Forest": "#1A6B9A",
    "XGBoost": "#3D8BB8",
    "QSVM": UNIR_AZUL,
    "QNN": "#5BB8D4",
    "VQC": "#8CCFE0",
}

plt.rcParams.update({
    "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
    "axes.edgecolor": UNIR_CLARO,
    "grid.color": UNIR_CLARO,
    "grid.linestyle": "--",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 150,
})


## Datos (mismo protocolo que Classification)


In [ ]:
# 2. Dataset preparation
SEED = 123
TEST_SIZE = 0.4
N_QUBITS = 4
N_CLASSES = 3

np.random.seed(SEED)
torch.manual_seed(SEED)

iris = datasets.load_iris()
X_raw = iris.data
y_raw = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=TEST_SIZE, random_state=SEED, stratify=y_raw
)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train)
X_test = np.clip(scaler.transform(X_test), 0, np.pi)

CLASSES = np.unique(y_train)
print(f"Train={len(X_train)}, Test={len(X_test)}, features={X_train.shape[1]}")


## Utilidades de evaluación


In [ ]:
# 3. Evaluation helpers
def classification_metrics(y_true, preds, scores):
    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, average="macro", zero_division=0)
    rec = recall_score(y_true, preds, average="macro", zero_division=0)
    f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    auc = roc_auc_score(label_binarize(y_true, classes=CLASSES), scores, average="macro")
    return {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "auc": float(auc),
    }


def timed(fn, *args, **kwargs):
    t0 = time.time()
    out = fn(*args, **kwargs)
    return out, time.time() - t0


results = {}  # model -> metrics dict


## 1) SVM RBF (clásico)


In [ ]:
# SVM
svm = SVC(kernel="rbf", random_state=42)
_, train_t = timed(svm.fit, X_train, y_train)
(_, infer_t) = timed(svm.predict, X_test)

def pack_sklearn(model, train_t, infer_t):
    tr_pred = model.predict(X_train)
    te_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        try:
            tr_scores = model.predict_proba(X_train)
            te_scores = model.predict_proba(X_test)
        except Exception:
            tr_scores = model.decision_function(X_train)
            te_scores = model.decision_function(X_test)
    else:
        tr_scores = model.decision_function(X_train)
        te_scores = model.decision_function(X_test)
    return {
        "family": "classical",
        "train": classification_metrics(y_train, tr_pred, tr_scores),
        "test": classification_metrics(y_test, te_pred, te_scores),
        "train_time_s": float(train_t),
        "inference_time_s": float(infer_t),
    }

results["SVM"] = pack_sklearn(svm, train_t, infer_t)
print(results["SVM"])


## 2) Random Forest (clásico)


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
_, train_t = timed(rf.fit, X_train, y_train)
(_, infer_t) = timed(rf.predict, X_test)
results["Random Forest"] = pack_sklearn(rf, train_t, infer_t)
print(results["Random Forest"])


## 3) XGBoost (clásico)


In [ ]:
# XGBoost
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    objective="multi:softprob",
    num_class=N_CLASSES,
    random_state=SEED,
    verbosity=0,
)
_, train_t = timed(xgb.fit, X_train, y_train)
(_, infer_t) = timed(xgb.predict, X_test)
results["XGBoost"] = pack_sklearn(xgb, train_t, infer_t)
print(results["XGBoost"])


## 4) QSVM — kernel ZZ (cuántico)


In [ ]:
# QSVM (Qiskit ZZFeatureMap + FidelityStatevectorKernel)
feature_map = zz_feature_map(feature_dimension=N_QUBITS, reps=2, entanglement="full")
quantum_kernel = FidelityStatevectorKernel(feature_map=feature_map)
qsvm = SVC(kernel=quantum_kernel.evaluate, random_state=42)

_, train_t = timed(qsvm.fit, X_train, y_train)
(_, infer_t) = timed(qsvm.predict, X_test)

# decision_function for AUC
tr_pred = qsvm.predict(X_train)
te_pred = qsvm.predict(X_test)
tr_scores = qsvm.decision_function(X_train)
te_scores = qsvm.decision_function(X_test)
results["QSVM"] = {
    "family": "quantum",
    "train": classification_metrics(y_train, tr_pred, tr_scores),
    "test": classification_metrics(y_test, te_pred, te_scores),
    "train_time_s": float(train_t),
    "inference_time_s": float(infer_t),
}
print(results["QSVM"])


## 5) QNN — PennyLane + PyTorch (cuántico)


In [ ]:
# QNN
dev_qnn = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_qnn, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.BasicEntanglerLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(qnode, {"weights": (6, N_QUBITS)})
        self.post = nn.Linear(N_QUBITS, N_CLASSES)

    def forward(self, x):
        return self.post(self.qlayer(x))


def train_qnn(model, loader, epochs=30, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()


X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)
loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

qnn_model = QNN()
_, train_t = timed(train_qnn, qnn_model, loader, 30, 0.01)

@torch.no_grad()
def qnn_eval(model, X_t, y_t):
    model.eval()
    logits = model(X_t)
    preds = torch.argmax(logits, dim=1).numpy()
    probs = nn.functional.softmax(logits, dim=1).numpy()
    return classification_metrics(y_t.numpy(), preds, probs), preds

(_, infer_t) = timed(lambda: qnn_model(X_test_t))
train_metrics, _ = qnn_eval(qnn_model, X_train_t, y_train_t)
test_metrics, _ = qnn_eval(qnn_model, X_test_t, y_test_t)

results["QNN"] = {
    "family": "quantum",
    "train": train_metrics,
    "test": test_metrics,
    "train_time_s": float(train_t),
    "inference_time_s": float(infer_t),
}
print(results["QNN"])


## 6) Variational Classifier (cuántico)


In [ ]:
# VQC (PennyLane)
dev_vqc = qml.device("default.qubit", wires=N_QUBITS)

def state_preparation(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

def layer(layer_weights):
    for wire in range(N_QUBITS):
        qml.Rot(*layer_weights[wire], wires=wire)
    for wires in ([0, 1], [1, 2], [2, 3], [3, 0]):
        qml.CNOT(wires=wires)

@qml.qnode(dev_vqc)
def circuit(weights, x):
    state_preparation(x)
    for lw in weights:
        layer(lw)
    return [qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1)), qml.expval(qml.PauliZ(2))]

def variational_classifier(weights, bias, x):
    return qml.math.stack(circuit(weights, x)) + bias

def cost(weights, bias, Xb, Yb):
    predictions = qml.math.stack([variational_classifier(weights, bias, x) for x in Xb])
    Yb = np.array(Yb, dtype=int)
    targets = np.eye(N_CLASSES)[Yb]
    return qml.math.mean((predictions - targets) ** 2)

# PennyLane numpy arrays for training
X_tr_v = pnp.array(X_train, requires_grad=False)
y_tr_v = pnp.array(y_train, requires_grad=False)

pnp.random.seed(SEED)
weights = 0.01 * pnp.random.randn(2, N_QUBITS, 3, requires_grad=True)
bias = pnp.zeros(N_CLASSES, requires_grad=True)
opt = NesterovMomentumOptimizer(0.1)
batch_size = 5

def train_vqc():
    global weights, bias
    for _ in range(100):
        batch_index = np.random.randint(0, len(X_tr_v), (batch_size,))
        weights, bias = opt.step(
            cost, weights, bias, X=X_tr_v[batch_index], Y=y_tr_v[batch_index]
        )

_, train_t = timed(train_vqc)

def vqc_predict_scores(X_data):
    scores = np.array([np.array(variational_classifier(weights, bias, x)) for x in X_data])
    preds = np.argmax(scores, axis=1)
    # shift scores to positive-ish probabilities via softmax
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    probs = e / e.sum(axis=1, keepdims=True)
    return preds, probs

t0 = time.time()
te_pred, te_scores = vqc_predict_scores(X_test)
infer_t = time.time() - t0
tr_pred, tr_scores = vqc_predict_scores(X_train)

results["VQC"] = {
    "family": "quantum",
    "train": classification_metrics(y_train, tr_pred, tr_scores),
    "test": classification_metrics(y_test, te_pred, te_scores),
    "train_time_s": float(train_t),
    "inference_time_s": float(infer_t),
}
print(results["VQC"])


## Guardar métricas


In [ ]:
# Persist metrics
metrics_path = METRICS_DIR / "iris_classification_metrics.json"
payload = {
    "protocol": {
        "dataset": "iris",
        "test_size": TEST_SIZE,
        "random_state": SEED,
        "scaler": "MinMaxScaler[0, pi]",
        "stratify": True,
    },
    "models": results,
}
metrics_path.write_text(json.dumps(payload, indent=2))
print("Saved:", metrics_path)

# Flat table for plotting
rows = []
for name, r in results.items():
    rows.append({
        "model": name,
        "family": r["family"],
        "split": "train",
        **r["train"],
        "train_time_s": r["train_time_s"],
        "inference_time_s": r["inference_time_s"],
    })
    rows.append({
        "model": name,
        "family": r["family"],
        "split": "test",
        **r["test"],
        "train_time_s": r["train_time_s"],
        "inference_time_s": r["inference_time_s"],
    })
df = pd.DataFrame(rows)
df_test = df[df["split"] == "test"].set_index("model").loc[ALL_MODELS].reset_index()
df_train = df[df["split"] == "train"].set_index("model").loc[ALL_MODELS].reset_index()
print(df_test[["model", "family", "accuracy", "precision", "recall", "f1", "auc", "train_time_s", "inference_time_s"]].to_string(index=False))


## Visualizaciones

Figuras pensadas para documento/slides: poca tinta, paleta UNIR, sin clutter.
Se guardan en `Notebooks/Results/figures/`.


In [ ]:
# Plot helpers
def savefig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    print("Saved:", path)
    plt.show()


def annotate_bars(ax, container, fmt="{:.2f}"):
    for bar in container:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.02, fmt.format(h),
                ha="center", va="bottom", fontsize=8, color=UNIR_GRIS)


### 1. Métricas de test por modelo


In [ ]:
# Fig 1: test metrics grouped bars
metric_cols = ["accuracy", "precision", "recall", "f1", "auc"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
x = np.arange(len(metric_labels))
width = 0.13

fig, ax = plt.subplots(figsize=(12, 5))
for i, model in enumerate(ALL_MODELS):
    vals = df_test.set_index("model").loc[model, metric_cols].to_numpy(dtype=float)
    bars = ax.bar(x + (i - 2.5) * width, vals, width, label=model, color=MODEL_COLORS[model], zorder=3)
    # only annotate accuracy-like dense charts lightly: skip per-bar text to avoid clutter

ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title("Iris — Test metrics by model", color=UNIR_NAVY, pad=12)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18))
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
savefig("iris_test_metrics_by_model.png")


### 2. Train vs Test (Accuracy y F1)


In [ ]:
# Fig 2: train vs test for Accuracy and F1
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, metric, title in zip(axs, ["accuracy", "f1"], ["Accuracy", "F1-score"]):
    x = np.arange(len(ALL_MODELS))
    width = 0.35
    train_vals = df_train.set_index("model").loc[ALL_MODELS, metric].to_numpy(dtype=float)
    test_vals = df_test.set_index("model").loc[ALL_MODELS, metric].to_numpy(dtype=float)
    b1 = ax.bar(x - width/2, train_vals, width, label="Train", color=UNIR_NAVY, zorder=3)
    b2 = ax.bar(x + width/2, test_vals, width, label="Test", color=UNIR_AZUL, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right")
    ax.set_ylim(0, 1.15)
    ax.set_title(title, color=UNIR_NAVY)
    ax.grid(axis="y", zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ax is axs[0]:
        ax.set_ylabel("Score")
        ax.legend(frameon=False, loc="lower right")

fig.suptitle("Iris — Train vs Test generalization", color=UNIR_NAVY, y=1.02)
fig.tight_layout()
savefig("iris_train_vs_test.png")


### 3. Clásicos vs cuánticos (test)


In [ ]:
# Fig 3: classical vs quantum aggregate on test
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(metric_labels))
width = 0.35
classical = df_test[df_test["family"] == "classical"][metric_cols].mean().to_numpy(dtype=float)
quantum = df_test[df_test["family"] == "quantum"][metric_cols].mean().to_numpy(dtype=float)
b1 = ax.bar(x - width/2, classical, width, label="Classical (mean)", color=UNIR_NAVY, zorder=3)
b2 = ax.bar(x + width/2, quantum, width, label="Quantum (mean)", color=UNIR_AZUL, zorder=3)
annotate_bars(ax, b1)
annotate_bars(ax, b2)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.2)
ax.set_ylabel("Score")
ax.set_title("Iris — Classical vs Quantum (test mean)", color=UNIR_NAVY)
ax.legend(frameon=False)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
savefig("iris_classical_vs_quantum.png")


### 4. Gap de generalización (train − test)


In [ ]:
# Fig 4: generalization gap
gap = (
    df_train.set_index("model").loc[ALL_MODELS, ["accuracy", "f1"]]
    - df_test.set_index("model").loc[ALL_MODELS, ["accuracy", "f1"]]
)
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(ALL_MODELS))
width = 0.35
b1 = ax.bar(x - width/2, gap["accuracy"], width, label="Acc gap", color=UNIR_NAVY, zorder=3)
b2 = ax.bar(x + width/2, gap["f1"], width, label="F1 gap", color=UNIR_AZUL, zorder=3)
ax.axhline(0, color=UNIR_GRIS, lw=1)
ax.set_xticks(x)
ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right")
ax.set_ylabel("Train − Test")
ax.set_title("Iris — Generalization gap", color=UNIR_NAVY)
ax.legend(frameon=False)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
savefig("iris_generalization_gap.png")


### 5. Tiempos de entrenamiento e inferencia


In [ ]:
# Fig 5: runtimes
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(ALL_MODELS))
train_times = df_test.set_index("model").loc[ALL_MODELS, "train_time_s"].to_numpy(dtype=float)
infer_times = df_test.set_index("model").loc[ALL_MODELS, "inference_time_s"].to_numpy(dtype=float)
colors = [MODEL_COLORS[m] for m in ALL_MODELS]

axs[0].bar(x, train_times, color=colors, zorder=3)
axs[0].set_xticks(x)
axs[0].set_xticklabels(ALL_MODELS, rotation=20, ha="right")
axs[0].set_ylabel("Seconds")
axs[0].set_title("Training time", color=UNIR_NAVY)
axs[0].set_yscale("log")
axs[0].grid(axis="y", zorder=0)
axs[0].spines["top"].set_visible(False)
axs[0].spines["right"].set_visible(False)

axs[1].bar(x, infer_times, color=colors, zorder=3)
axs[1].set_xticks(x)
axs[1].set_xticklabels(ALL_MODELS, rotation=20, ha="right")
axs[1].set_ylabel("Seconds")
axs[1].set_title("Inference time", color=UNIR_NAVY)
axs[1].set_yscale("log")
axs[1].grid(axis="y", zorder=0)
axs[1].spines["top"].set_visible(False)
axs[1].spines["right"].set_visible(False)

fig.suptitle("Iris — Computational cost", color=UNIR_NAVY, y=1.02)
fig.tight_layout()
savefig("iris_runtimes.png")


### 6. Heatmap resumen (test)


In [ ]:
# Fig 6: heatmap of test metrics
mat = df_test.set_index("model").loc[ALL_MODELS, metric_cols]
fig, ax = plt.subplots(figsize=(8, 4.2))
im = ax.imshow(mat.to_numpy(dtype=float), cmap="Blues", vmin=0.5, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(metric_labels)))
ax.set_xticklabels(metric_labels)
ax.set_yticks(range(len(ALL_MODELS)))
ax.set_yticklabels(ALL_MODELS)
ax.set_title("Iris — Test metrics heatmap", color=UNIR_NAVY)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat.iloc[i, j]:.2f}", ha="center", va="center", color="black", fontsize=9)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=8)
fig.tight_layout()
savefig("iris_test_heatmap.png")


## Tabla final para documento


In [ ]:
# Final markdown-friendly table
summary = df_test[["model", "family", "accuracy", "f1", "auc", "train_time_s"]].copy()
summary["acc_gap"] = (
    df_train.set_index("model").loc[summary["model"], "accuracy"].to_numpy()
    - summary["accuracy"].to_numpy()
)
summary = summary.round({"accuracy": 4, "f1": 4, "auc": 4, "train_time_s": 3, "acc_gap": 4})
summary_path = METRICS_DIR / "iris_classification_summary.csv"
summary.to_csv(summary_path, index=False)
print(summary.to_string(index=False))
print("\nSaved:", summary_path)
print("\nFigures:")
for p in sorted(FIGURES_DIR.glob("iris_*.png")):
    print(" -", p)
